Rešava se problem ranog prepoznavanja da li će student da diplomira ili odustane. U pitanju je klasifikacio ni problem za 3 klase (odustao, diplomnirao i još studira).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer

In [ ]:
import importlib.util
import sys

def package_install(package_name):
    if importlib.util.find_spec(package_name) is None:
        print(f'{package_name} nije instaliran. Instaliram...')
        !{sys.executable} -m pip install {package_name}
    else:
        print(f'{package_name} je već instaliran.')

In [ ]:
package_install('ucimlrepo')
package_install('imbalanced-learn')

In [ ]:
from ucimlrepo import fetch_ucirepo
from imblearn.over_sampling import SMOTE

# fetch dataset
predict_students_dropout_and_academic_success = fetch_ucirepo(id=697)

# data (as pandas dataframes)
X = predict_students_dropout_and_academic_success.data.features
y = predict_students_dropout_and_academic_success.data.targets


In [ ]:
X = X.drop_duplicates()
y=y.loc[X.index]


In [ ]:
y.head()

In [ ]:
opis = ''
oblik = f'Baza ima {X.shape[0]} uzoraka i {X.shape[1]} obeležje.'
print(oblik)
opis += oblik

Jedan uzorak predstavlja podatke jednog studenta.

In [ ]:
X.head()

In [ ]:
obelezja = f'Obeležja u tabeli su {', '.join(X.columns.values)}.'
print(obelezja)
opis += obelezja

Obeležja u tabeli su Bračni status, Način prijave, Broj na listi želja, Studijski program, Prisustvo, Prethodna kvalifikacija, Ocena iz prethodne kvalifikacije, Nacionalnost, Kvalifikacija majke, Kvalifikacija oca, Zanimanje majke, Zanimanje oca, Ocena na upisu, Preseljen zbog studija, Posebne obrazovne potrebe, Neizmirena dugovanja prema fakultetu, Školarina ažurna, Pol, Stipendista, Starost pri upisu, Strani student, Nastavne jedinice 1. semestar (priznate), Nastavne jedinice 1. semestar (upisane), Nastavne jedinice 1. semestar (provere/izlasci), Nastavne jedinice 1. semestar (položene), Nastavne jedinice 1. semestar (ocena), Nastavne jedinice 1. semestar (bez provera), Nastavne jedinice 2. semestar (priznate), Nastavne jedinice 2. semestar (upisane), Nastavne jedinice 2. semestar (provere/izlasci), Nastavne jedinice 2. semestar (položene), Nastavne jedinice 2. semestar (ocena), Nastavne jedinice 2. semestar (bez provera), Stopa nezaposlenosti, Stopa inflacije, BDP.

In [ ]:
numerička = X.columns[[2,6,12,19] + list(range(21, 36))]
print(numerička)

In [ ]:
kategorička = X.columns.drop(numerička)
print(kategorička)

In [ ]:
X_kategorička = X[kategorička].copy()
X_numerička = X[numerička].copy()

In [ ]:
categorical_value_names = {
    "Marital Status": {
        1: "single",
        2: "married",
        3: "widower",
        4: "divorced",
        5: "facto union",
        6: "legally separated",
    },
    "Application mode": {
        1: "1st phase - general contingent",
        2: "Ordinance No. 612/93",
        5: "1st phase - special contingent (Azores Island)",
        7: "Holders of other higher courses",
        10: "Ordinance No. 854-B/99",
        15: "International student (bachelor)",
        16: "1st phase - special contingent (Madeira Island)",
        17: "2nd phase - general contingent",
        18: "3rd phase - general contingent",
        26: "Ordinance No. 533-A/99, item b2) (Different Plan)",
        27: "Ordinance No. 533-A/99, item b3) (Other Institution)",
        39: "Over 23 years old",
        42: "Transfer",
        43: "Change of course",
        44: "Technological specialization diploma holders",
        51: "Change of institution/course",
        53: "Short cycle diploma holders",
        57: "Change of institution/course (International)",
    },
    "Course": {
        33: "Biofuel Production Technologies",
        171: "Animation and Multimedia Design",
        8014: "Social Service (evening attendance)",
        9003: "Agronomy",
        9070: "Communication Design",
        9085: "Veterinary Nursing",
        9119: "Informatics Engineering",
        9130: "Equinculture",
        9147: "Management",
        9238: "Social Service",
        9254: "Tourism",
        9500: "Nursing",
        9556: "Oral Hygiene",
        9670: "Advertising and Marketing Management",
        9773: "Journalism and Communication",
        9853: "Basic Education",
        9991: "Management (evening attendance)",
    },
    "Daytime/evening attendance": {
        0: "evening",
        1: "daytime",
    },
    "Previous qualification": {
        1: "Secondary education",
        2: "Higher education - bachelor's degree",
        3: "Higher education - degree",
        4: "Higher education - master's",
        5: "Higher education - doctorate",
        6: "Frequency of higher education",
        9: "12th year of schooling - not completed",
        10: "11th year of schooling - not completed",
        12: "Other - 11th year of schooling",
        14: "10th year of schooling",
        15: "10th year of schooling - not completed",
        19: "Basic education 3rd cycle (9th/10th/11th year) or equiv.",
        38: "Basic education 2nd cycle (6th/7th/8th year) or equiv.",
        39: "Technological specialization course",
        40: "Higher education - degree (1st cycle)",
        42: "Professional higher technical course",
        43: "Higher education - master (2nd cycle)",
    },
    "Nacionality": {
        1: "Portuguese",
        2: "German",
        6: "Spanish",
        11: "Italian",
        13: "Dutch",
        14: "English",
        17: "Lithuanian",
        21: "Angolan",
        22: "Cape Verdean",
        24: "Guinean",
        25: "Mozambican",
        26: "Santomean",
        32: "Turkish",
        41: "Brazilian",
        62: "Romanian",
        100: "Moldovan",
        101: "Mexican",
        103: "Ukrainian",
        105: "Russian",
        108: "Cuban",
        109: "Colombian",
    },
    "Mother's qualification": {
        1: "Secondary Education - 12th Year of Schooling or Eq.",
        2: "Higher Education - Bachelor's Degree",
        3: "Higher Education - Degree",
        4: "Higher Education - Master's",
        5: "Higher Education - Doctorate",
        6: "Frequency of Higher Education",
        9: "12th Year of Schooling - Not Completed",
        10: "11th Year of Schooling - Not Completed",
        11: "7th Year (Old)",
        12: "Other - 11th Year of Schooling",
        14: "10th Year of Schooling",
        18: "General commerce course",
        19: "Basic Education 3rd Cycle (9th/10th/11th Year) or Equiv.",
        22: "Technical-professional course",
        26: "7th year of schooling",
        27: "2nd cycle of the general high school course",
        29: "9th Year of Schooling - Not Completed",
        30: "8th year of schooling",
        34: "Unknown",
        35: "Can't read or write",
        36: "Can read without having a 4th year of schooling",
        37: "Basic education 1st cycle (4th/5th year) or equiv.",
        38: "Basic Education 2nd Cycle (6th/7th/8th Year) or Equiv.",
        39: "Technological specialization course",
        40: "Higher education - degree (1st cycle)",
        41: "Specialized higher studies course",
        42: "Professional higher technical course",
        43: "Higher Education - Master (2nd cycle)",
        44: "Higher Education - Doctorate (3rd cycle)",
    },
    "Father's qualification": {
        1: "Secondary Education - 12th Year of Schooling or Eq.",
        2: "Higher Education - Bachelor's Degree",
        3: "Higher Education - Degree",
        4: "Higher Education - Master's",
        5: "Higher Education - Doctorate",
        6: "Frequency of Higher Education",
        9: "12th Year of Schooling - Not Completed",
        10: "11th Year of Schooling - Not Completed",
        11: "7th Year (Old)",
        12: "Other - 11th Year of Schooling",
        13: "2nd year complementary high school course",
        14: "10th Year of Schooling",
        18: "General commerce course",
        19: "Basic Education 3rd Cycle (9th/10th/11th Year) or Equiv.",
        20: "Complementary High School Course",
        22: "Technical-professional course",
        25: "Complementary High School Course - not concluded",
        26: "7th year of schooling",
        27: "2nd cycle of the general high school course",
        29: "9th Year of Schooling - Not Completed",
        30: "8th year of schooling",
        31: "General Course of Administration and Commerce",
        33: "Supplementary Accounting and Administration",
        34: "Unknown",
        35: "Can't read or write",
        36: "Can read without having a 4th year of schooling",
        37: "Basic education 1st cycle (4th/5th year) or equiv.",
        38: "Basic Education 2nd Cycle (6th/7th/8th Year) or Equiv.",
        39: "Technological specialization course",
        40: "Higher education - degree (1st cycle)",
        41: "Specialized higher studies course",
        42: "Professional higher technical course",
        43: "Higher Education - Master (2nd cycle)",
        44: "Higher Education - Doctorate (3rd cycle)",
    },
    "Mother's occupation": {
        0: "Student",
        1: "Representatives of the Legislative Power and Executive Bodies, Directors, Directors and Executive Managers",
        2: "Specialists in Intellectual and Scientific Activities",
        3: "Intermediate Level Technicians and Professions",
        4: "Administrative staff",
        5: "Personal Services, Security and Safety Workers and Sellers",
        6: "Farmers and Skilled Workers in Agriculture, Fisheries and Forestry",
        7: "Skilled Workers in Industry, Construction and Craftsmen",
        8: "Installation and Machine Operators and Assembly Workers",
        9: "Unskilled Workers",
        10: "Armed Forces Professions",
        90: "Other Situation",
        99: "Blank",
        122: "Health professionals",
        123: "Teachers",
        125: "Specialists in information and communication technologies (ICT)",
        131: "Intermediate level science and engineering technicians and professions",
        132: "Technicians and professionals, of intermediate level of health",
        134: "Intermediate level technicians from legal, social, sports, cultural and similar services",
        141: "Office workers, secretaries in general and data processing operators",
        143: "Data, accounting, statistical, financial services and registry-related operators",
        144: "Other administrative support staff",
        151: "Personal service workers",
        152: "Sellers",
        153: "Personal care workers and the like",
        171: "Skilled construction workers and the like, except electricians",
        173: "Skilled workers in printing, precision instrument manufacturing, jewelers, artisans and the like",
        175: "Workers in food processing, woodworking, clothing and other industries and crafts",
        191: "Cleaning workers",
        192: "Unskilled workers in agriculture, animal production, fisheries and forestry",
        193: "Unskilled workers in extractive industry, construction, manufacturing and transport",
        194: "Meal preparation assistants",
    },
    "Father's occupation": {
        0: "Student",
        1: "Representatives of the Legislative Power and Executive Bodies, Directors, Directors and Executive Managers",
        2: "Specialists in Intellectual and Scientific Activities",
        3: "Intermediate Level Technicians and Professions",
        4: "Administrative staff",
        5: "Personal Services, Security and Safety Workers and Sellers",
        6: "Farmers and Skilled Workers in Agriculture, Fisheries and Forestry",
        7: "Skilled Workers in Industry, Construction and Craftsmen",
        8: "Installation and Machine Operators and Assembly Workers",
        9: "Unskilled Workers",
        10: "Armed Forces Professions",
        90: "Other Situation",
        99: "Blank",
        122: "Health professionals",
        123: "Teachers",
        125: "Specialists in information and communication technologies (ICT)",
        131: "Intermediate level science and engineering technicians and professions",
        132: "Technicians and professionals, of intermediate level of health",
        134: "Intermediate level technicians from legal, social, sports, cultural and similar services",
        141: "Office workers, secretaries in general and data processing operators",
        143: "Data, accounting, statistical, financial services and registry-related operators",
        144: "Other administrative support staff",
        151: "Personal service workers",
        152: "Sellers",
        153: "Personal care workers and the like",
        171: "Skilled construction workers and the like, except electricians",
        173: "Skilled workers in printing, precision instrument manufacturing, jewelers, artisans and the like",
        175: "Workers in food processing, woodworking, clothing and other industries and crafts",
        191: "Cleaning workers",
        192: "Unskilled workers in agriculture, animal production, fisheries and forestry",
        193: "Unskilled workers in extractive industry, construction, manufacturing and transport",
        194: "Meal preparation assistants",
    },
    "Displaced": {0: "no", 1: "yes"},
    "Educational special needs": {0: "no", 1: "yes"},
    "Debtor": {0: "no", 1: "yes"},
    "Tuition fees up to date": {0: "no", 1: "yes"},
    "Gender": {0: "female", 1: "male"},
    "Scholarship holder": {0: "no", 1: "yes"},
    "International": {0: "no", 1: "yes"},
}



In [ ]:
categorical_value_names = {
    "Marital Status": {
        1: "Single",
        2: "Married",
        3: "Widowed",
        4: "Divorced",
        5: "Facto Union",
        6: "Legally Separated",
    },
    "Application mode": {
        1: "1st Phase - General",
        2: "Ord. 612/93",
        5: "1st Phase - Spec. (Azores)",
        7: "Holders of HE Degree",
        10: "Ord. 854-B/99",
        15: "Int. Student (BSc)",
        16: "1st Phase - Spec. (Madeira)",
        17: "2nd Phase - General",
        18: "3rd Phase - General",
        26: "Ord. 533-A/99 (B2)",
        27: "Ord. 533-A/99 (B3)",
        39: "Over 23 Years Old",
        42: "Transfer",
        43: "Course Change",
        44: "Tech. Spec. Diploma",
        51: "Inst./Course Change",
        53: "Short Cycle Diploma",
        57: "Int. Change Inst./Course",
    },
    "Course": {
        33: "Biofuel Production Tech.",
        171: "Animation & Multimedia Design",
        8014: "Social Service (Evening)",
        9003: "Agronomy",
        9070: "Communication Design",
        9085: "Veterinary Nursing",
        9119: "Informatics Engineering",
        9130: "Equinculture",
        9147: "Management",
        9238: "Social Service (Day)",
        9254: "Tourism",
        9500: "Nursing",
        9556: "Oral Hygiene",
        9670: "Advertising & Marketing",
        9773: "Journalism & Comm.",
        9853: "Basic Education",
        9991: "Management (Evening)",
    },
    "Daytime/evening attendance": {
        0: "Evening",
        1: "Daytime",
    },
    "Previous qualification": {
        1: "Secondary Education",
        2: "Higher Ed. - Bachelor",
        3: "Higher Ed. - Degree",
        4: "Higher Ed. - Master",
        5: "Higher Ed. - Doctorate",
        6: "Incomplete Higher Ed.",
        9: "12th Year (Incomplete)",
        10: "11th Year (Incomplete)",
        12: "11th Year (Completed)",
        14: "10th Year (Completed)",
        15: "10th Year (Incomplete)",
        19: "Basic Ed. 3rd Cycle",
        38: "Basic Ed. 2nd Cycle",
        39: "Tech. Spec. Course",
        40: "Higher Ed. - 1st Cycle",
        42: "Prof. Higher Tech. Course",
        43: "Higher Ed. - 2nd Cycle",
    },
    "Nacionality": {
        1: "Portuguese", 2: "German", 6: "Spanish", 11: "Italian", 
        13: "Dutch", 14: "English", 17: "Lithuanian", 21: "Angolan", 
        22: "Cape Verdean", 24: "Guinean", 25: "Mozambican", 26: "Santomean", 
        32: "Turkish", 41: "Brazilian", 62: "Romanian", 100: "Moldovan", 
        101: "Mexican", 103: "Ukrainian", 105: "Russian", 108: "Cuban", 109: "Colombian",
    },
    "Mother's qualification" : { 
        1: "Secondary Education",
        2: "Higher Ed. - Bachelor",
        3: "Higher Ed. - Degree",
        4: "Higher Ed. - Master",
        5: "Higher Ed. - Doctorate",
        6: "Incomplete Higher Ed.",
        9: "12th Year (Incomp.)",
        10: "11th Year (Incomp.)",
        11: "7th Year (Old System)",
        12: "11th Year",
        14: "10th Year",
        18: "General Commerce",
        19: "Basic Ed. 3rd Cycle",
        22: "Tech.-Professional",
        26: "7th Year",
        27: "HS 2nd Cycle",
        29: "9th Year (Incomp.)",
        30: "8th Year",
        34: "Unknown",
        35: "Illiterate",
        36: "Basic Literacy",
        37: "Basic Ed. 1st Cycle",
        38: "Basic Ed. 2nd Cycle",
        39: "Tech. Spec. Course",
        40: "Higher Ed. 1st Cycle",
        41: "Spec. Higher Studies",
        42: "Prof. Higher Tech.",
        43: "Higher Ed. 2nd Cycle",
        44: "Higher Ed. 3rd Cycle",
    },
    "Father's qualification": {
        1: "Secondary Education",
        2: "Higher Ed. - Bachelor",
        3: "Higher Ed. - Degree",
        4: "Higher Ed. - Master",
        5: "Higher Ed. - Doctorate",
        6: "Incomplete Higher Ed.",
        9: "12th Year (Incomp.)",
        10: "11th Year (Incomp.)",
        11: "7th Year (Old System)",
        12: "11th Year",
        14: "10th Year",
        18: "General Commerce",
        19: "Basic Ed. 3rd Cycle",
        22: "Tech.-Professional",
        26: "7th Year",
        27: "HS 2nd Cycle",
        29: "9th Year (Incomp.)",
        30: "8th Year",
        34: "Unknown",
        35: "Illiterate",
        36: "Basic Literacy",
        37: "Basic Ed. 1st Cycle",
        38: "Basic Ed. 2nd Cycle",
        39: "Tech. Spec. Course",
        40: "Higher Ed. 1st Cycle",
        41: "Spec. Higher Studies",
        42: "Prof. Higher Tech.",
        43: "Higher Ed. 2nd Cycle",
        44: "Higher Ed. 3rd Cycle",
    },
    "Mother's occupation": { # Skraćeno za majku i oca
        0: "Student",
        1: "Exec. Managers / Directors",
        2: "Intellectual / Scientific",
        3: "Intermediate Tech. Prof.",
        4: "Administrative Staff",
        5: "Services & Sales",
        6: "Agri. / Fishery Skilled",
        7: "Industry / Craft Skilled",
        8: "Machine / Assembly Ops.",
        9: "Unskilled Workers",
        10: "Armed Forces",
        90: "Other",
        99: "N/A",
        122: "Health Professionals",
        123: "Teachers",
        125: "ICT Specialists",
        131: "Sci. & Eng. Technicians",
        132: "Health Technicians",
        134: "Legal / Social Technicians",
        141: "Office / Data Operators",
        143: "Financial / Registry Ops.",
        144: "Admin. Support Staff",
        151: "Personal Service",
        152: "Sales Personnel",
        153: "Personal Care Workers",
        171: "Construction Workers",
        173: "Printing / Precision Crafts",
        175: "Food / Wood / Textile Ind.",
        191: "Cleaning Personnel",
        192: "Unskilled Agri. / Forestry",
        193: "Unskilled Mfg. / Transport",
        194: "Food Prep. Assistants",
    },
    "Father's occupation": { # Skraćeno za majku i oca
        0: "Student",
        1: "Exec. Managers / Directors",
        2: "Intellectual / Scientific",
        3: "Intermediate Tech. Prof.",
        4: "Administrative Staff",
        5: "Services & Sales",
        6: "Agri. / Fishery Skilled",
        7: "Industry / Craft Skilled",
        8: "Machine / Assembly Ops.",
        9: "Unskilled Workers",
        10: "Armed Forces",
        90: "Other",
        99: "N/A",
        122: "Health Professionals",
        123: "Teachers",
        125: "ICT Specialists",
        131: "Sci. & Eng. Technicians",
        132: "Health Technicians",
        134: "Legal / Social Technicians",
        141: "Office / Data Operators",
        143: "Financial / Registry Ops.",
        144: "Admin. Support Staff",
        151: "Personal Service",
        152: "Sales Personnel",
        153: "Personal Care Workers",
        171: "Construction Workers",
        173: "Printing / Precision Crafts",
        175: "Food / Wood / Textile Ind.",
        191: "Cleaning Personnel",
        192: "Unskilled Agri. / Forestry",
        193: "Unskilled Mfg. / Transport",
        194: "Food Prep. Assistants",
    },
    "Displaced": {0: "No", 1: "Yes"},
    "Educational special needs": {0: "No", 1: "Yes"},
    "Debtor": {0: "No", 1: "Yes"},
    "Tuition fees up to date": {0: "No", 1: "Yes"},
    "Gender": {0: "Female", 1: "Male"},
    "Scholarship holder": {0: "No", 1: "Yes"},
    "International": {0: "No", 1: "Yes"},
}

def label_categorical_values(df, mappings):
    result = df.copy()
    for column, mapping in mappings.items():
        if column in result.columns:
            result[column] = result[column].replace(mapping)
    return result

In [ ]:
dict_kateg = pd.Series({k:X_kategorička[k].nunique() for k in X_kategorička.columns})
print(dict_kateg)


Ima 19 numeričkih i 17 kategoričkih obeležja.

In [ ]:
print(f'Obeležja sa najmanje kategorija ({dict_kateg.min()}): { ', '.join([k[0] for k in dict_kateg.items() if k[1]==dict_kateg.min()]) }')

In [ ]:
print(f'Obeležja sa najviše kategorija ({dict_kateg.max()}): { ', '.join([k[0] for k in dict_kateg.items() if k[1]==dict_kateg.max()]) }')

In [ ]:
print(y.value_counts())
print(y.value_counts(normalize=True)*100)
print(y.value_counts(normalize=True).sum()*100)

In [ ]:
missing_values_count = X.isnull().sum()
print('Broj nedostajućih vrednosti po obeležjima:')
print(missing_values_count)

In [ ]:
X_numerička.hist(figsize=(17, 17), bins=30)
plt.autoscale()
plt.show()
print(numerička)

Nema obeležja za izbacivanje i nema nedostajućih vrednosti.

In [ ]:
X_numerička.describe([0, 0.05, 0.95,0.98, 0.97, 0.99,0.995])

In [ ]:
#X['Previous qualification (grade)'] = X['Previous qualification (grade)'].clip(upper=X['Previous qualification (grade)'].quantile(0.999))
#X['Age at enrollment'] = X['Age at enrollment'].clip(upper=X['Age at enrollment'].quantile(0.95))
long_tail = ['Age at enrollment', 'Application order']
X[long_tail] = X[long_tail].clip(upper=X[long_tail].quantile(0.90), axis=1)
long_tail_s = ['Curricular units 1st sem (credited)', 'Curricular units 2nd sem (credited)']
X[long_tail_s] = X[long_tail_s].clip(upper=X[long_tail_s].quantile(0.95), axis=1)
long_tail_xs = ['Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (without evaluations)']
X[long_tail_xs] = X[long_tail_xs].clip(upper=X[long_tail_xs].quantile(0.99), axis=1)

tail = ['Curricular units 1st sem (enrolled)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 2nd sem (approved)']
X[tail] = X[tail].clip(upper=X[tail].quantile(0.97), axis=1)
#X['Admission grade'] = X['Admission grade'].clip(lower=X['Admission grade'].quantile(0.01), upper=X['Admission grade'].quantile(0.99))
cols = list(X_numerička.columns[4:8])+ list(X_numerička.columns[9:13]) + list(X_numerička.columns[14:16])
#X[numerička] = X[numerička].clip(upper=X[numerička].quantile(0.98), axis=1)


In [ ]:
X[numerička].hist(figsize=(17, 17), bins=30)
plt.autoscale()
plt.show()

In [ ]:
# 1. Provera duplikata
full_df = pd.concat([X, y], axis=1)
print(f"Broj dupliranih redova: {full_df.duplicated().sum()}")

# 2. Tipovi podataka
print("\nTipovi podataka u X:")
print(X.dtypes.value_counts())

# 3. Balans klasa
print("\nBalans klasa (procentualni udeo):")
balans = y['Target'].value_counts(normalize=True) * 100
print(balans.map('{:f}%'.format))

# 4. Korelacija (Popravljen deo)
temp_df = pd.concat([X, y], axis=1)
# Pretvaramo Target u brojeve: Graduate=0, Dropout=1, Enrolled=2 (ili slično)
temp_df['Target_encoded'] = temp_df['Target'].astype('category').cat.codes

print("\nTop 5 obeležja sa najjačim uticajem na ishod:")
# Ključna promena: numeric_only=True preskače tekstualnu kolonu 'Target'
correlations = temp_df.corr(numeric_only=True)['Target_encoded'].abs().sort_values(ascending=False)

# Prikazujemo top 5 (preskačemo 0 jer je to korelacija Target-a sa samim sobom)
print(correlations.iloc[1:6])

In [ ]:
def plot_kategoricka():
    m = 4 if int(len(kategorička)**1/2) > 4 else int(len(kategorička)**1/2)
    n = math.ceil(len(numerička)/2)
    plt.figure(figsize=(m * 6, n * 6)) 
    for k in kategorička:
        plt.subplot(n, m, list(kategorička).index(k)+1)
        sns.countplot(y=X[k], hue=y['Target'])
        #plt.tick_params(axis='x', rotation=90)
        plt.tight_layout()
        plt.autoscale()
    plt.show()

X = label_categorical_values(X, categorical_value_names)
#plot_kategoricka()

In [ ]:
def other_categories(df, columns, percentile, min_rare_count=3, other_label='other'):
    if not 0 < percentile < 1:
        raise ValueError("percentile must be in decimal form, between 0 and 1, for example 0.05")
    if min_rare_count < 1:
        raise ValueError("min_rare_count must be >= 1")

    result = df.copy()
    rare_values_by_column = {}

    for column in columns:
        value_frequencies = result[column].value_counts(normalize=True, dropna=True)
        rare_values = value_frequencies[value_frequencies < percentile].index

        # Group only if there are enough rare categories in this column.
        if len(rare_values) >= min_rare_count:
            rare_values_by_column[column] = list(rare_values)
            result[column] = result[column].where(~result[column].isin(rare_values), other_label)
        else:
            rare_values_by_column[column] = []

    return result, rare_values_by_column


In [ ]:
for col in X_kategorička.columns:
    proc = (
        X_kategorička[col]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .sort_values(ascending=False)
        .round(2)
    )

    # Ako postoji mapa, prikaži opis umesto šifre
    if col in categorical_value_names:
        prikaz = pd.DataFrame({
            "vrednost": [categorical_value_names[col].get(v, v) for v in proc.index],
            "procenat": proc.values
        })
    else:
        prikaz = pd.DataFrame({
            "vrednost": proc.index,
            "procenat": proc.values
        })

    print(f"\n--- {col} ---")
    print(prikaz.to_string(index=False))

In [ ]:
X , rare= other_categories(X, kategorička, 0.02)

In [ ]:
print(rare)
plot_kategoricka()



In [ ]:
X = pd.get_dummies(X, drop_first=True, dtype=int, columns=kategorička)
y = y['Target'].map({'Graduate': 0, 'Dropout': 1, 'Enrolled': 2})

In [ ]:
X_trainval, X_testval, y_trainval, y_testval = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval)

In [ ]:
balancer = SMOTE()
X_train_balanced, y_train_balanced = balancer.fit_resample(X_train, y_train)
print("Balans klasa nakon SMOTE:")
print(y_train_balanced.value_counts(normalize=True))


In [ ]:
# Todo: pipeline: odvojeno skaliranje numeričkih i kategoričkih, modeli (knn + pca, mlp, linreg), evaluacija
num_cols = [col for col in numerička if col in X_train_balanced.columns]
cat_cols = [col for col in X_train_balanced.columns if col not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', 'passthrough', cat_cols)
    ]
)

# Pipeline sa KNN i PCA (skaliranje samo numeričkih kolona)
pipeline_knn_pca = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('knn', KNeighborsClassifier())
])

# GridSearchCV za KNN i PCA
param_grid_knn_pca = {
    'knn__n_neighbors': [3, 5, 7],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan']
}

grid_search_knn_pca = GridSearchCV(
    pipeline_knn_pca,
    param_grid_knn_pca,
    cv=StratifiedKFold(n_splits=5),
    scoring='accuracy'
 )
grid_search_knn_pca.fit(X_train_balanced, y_train_balanced)

print('Numeričkih kolona:', len(num_cols))
print('Kategoričkih kolona (one-hot):', len(cat_cols))
print('Best parameters for KNN + PCA:', grid_search_knn_pca.best_params_)
print('Best cross-validation accuracy for KNN + PCA:', grid_search_knn_pca.best_score_)